## Load data

In [1]:
import pandas as pd

df = pd.read_csv("IMDB Dataset.csv")

# Convert labels to 0/1
df['sentiment'] = df['sentiment'].map({
    'positive': 1,
    'negative': 0
})

print(df.head())
print(df['sentiment'].value_counts())

                                              review  sentiment
0  One of the other reviewers has mentioned that ...          1
1  A wonderful little production. <br /><br />The...          1
2  I thought this was a wonderful way to spend ti...          1
3  Basically there's a family where a little boy ...          0
4  Petter Mattei's "Love in the Time of Money" is...          1
sentiment
1    25000
0    25000
Name: count, dtype: int64


## Clean the Text

In [2]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"<.*?>", "", text)   # remove HTML
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    return text

## Tokenization

In [3]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_words = 10000
max_length = 100

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(df['review'])

sequences = tokenizer.texts_to_sequences(df['review'])
padded = pad_sequences(sequences, maxlen=max_length, padding='post')

## Split and Test

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    padded, df['sentiment'], test_size=0.2, random_state=42
)

## Build the Model

In [5]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

model = Sequential([
    Embedding(max_words, 64, input_length=max_length),
    LSTM(64),
    Dense(1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 100, 64)           640000    
                                                                 
 lstm (LSTM)                 (None, 64)                33024     
                                                                 
 dense (Dense)               (None, 1)                 65        
                                                                 
Total params: 673089 (2.57 MB)
Trainable params: 673089 (2.57 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


## Fine Tuning

In [6]:
history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=128,
    validation_data=(X_test, y_test)
)

Epoch 1/5
313/313 [==============================] - 92s 278ms/step - loss: 0.4094 - accuracy: 0.8046 - val_loss: 0.3165 - val_accuracy: 0.8584
Epoch 2/5
313/313 [==============================] - 88s 280ms/step - loss: 0.2682 - accuracy: 0.8908 - val_loss: 0.3364 - val_accuracy: 0.8593
Epoch 3/5
313/313 [==============================] - 84s 268ms/step - loss: 0.2290 - accuracy: 0.9080 - val_loss: 0.3353 - val_accuracy: 0.8594
Epoch 4/5
313/313 [==============================] - 83s 264ms/step - loss: 0.1840 - accuracy: 0.9297 - val_loss: 0.3767 - val_accuracy: 0.8561
Epoch 5/5
313/313 [==============================] - 82s 261ms/step - loss: 0.1481 - accuracy: 0.9453 - val_loss: 0.4261 - val_accuracy: 0.8497


## Test Predictions

In [7]:
def predict_sentiment(text):
    text = clean_text(text)
    seq = tokenizer.texts_to_sequences([text])
    padded_seq = pad_sequences(seq, maxlen=max_length, padding='post')
    
    pred = model.predict(padded_seq)[0][0]
    
    if pred >= 0.5:
        return "Positive", float(pred)
    else:
        return "Negative", float(pred)

In [18]:
print(predict_sentiment("I Love this Movie"))
print(predict_sentiment("I dont like this Movie"))

1/1 [==============================] - 0s 60ms/step
('Positive', 0.994282603263855)
1/1 [==============================] - 0s 48ms/step
('Positive', 0.6385000944137573)


## Turn into Chatbot Brain

In [21]:
def chatbot_response(text):
    text_lower = text.lower()

    negative_words = ["hate", "bad", "terrible", "awful", "worst", "boring"]
    positive_words = ["love", "amazing", "great", "awesome", "fantastic"]

    if any(word in text_lower for word in negative_words):
        return "😅 That doesn’t sound great. What didn’t you like?"

    if any(word in text_lower for word in positive_words):
        return "😊 Glad you liked it!"

    # fallback to model
    label = predict_sentiment(text)

    if label == "Positive":
        return "😊 Glad you liked it!"
    else:
        return "😅 Sorry it wasn’t great. What went wrong?"

## Save the Model

In [22]:
model.save("sentiment_model.keras")

# Save the Tokenizer
import pickle

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

# Save the Configuration
config = {
    "max_length": max_length
}

with open("config.pkl", "wb") as f:
    pickle.dump(config, f)



In [13]:
import random

positive_responses = [
    "Glad you liked it! 🎬",
    "Nice! Sounds like a great movie 😄",
    "Awesome, I might watch it too 👀"
]

negative_responses = [
    "Ouch 😅 that sounds disappointing",
    "Yikes… what didn’t you like?",
    "That’s rough 😬 was it the acting or story?"
]

def chatbot_response(user_input):
    label, score = predict_sentiment(user_input)
    
    if label == "Positive":
        return random.choice(positive_responses)
    else:
        return random.choice(negative_responses)

In [14]:
while True:
    user_input = input("You: This was boring and too long ")
    
    if user_input.lower() in ["exit", "quit"]:
        print("Bot: Bye! 👋")
        break
    
    response = chatbot_response(user_input)
    print("Bot:", response)

You: This was boring and too long  boring


1/1 [==============================] - 0s 102ms/step
Bot: Yikes… what didn’t you like?


You: This was boring and too long  exit


Bot: Bye! 👋


In [15]:
import tensorflow as tf
print(tf.__version__)

2.13.0
